# Real shadow data: feeding the UCB diagnostic from any provider

This notebook is the landing pad for the question "I have random-Pauli shadow data from `<provider>`; what do I do with it?"

The library does not generate shadow shots. You bring them. The contract is a single iterable of `(basis, outcomes)` tuples where:
- `basis` is a tuple of `"X"` / `"Y"` / `"Z"` of length `n_qubits`,
- `outcomes` is a tuple of $\pm 1$ of length `n_qubits`.

Below, the `fetch_shadow_data(provider=...)` function returns shots in that format from any backend. The simulator path is always available and runs in seconds. The three hardware-provider paths are stubs you fill in if you have credentials.

**No API key is required to run this notebook end to end.** The default `provider="simulator"` path is fully classical.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh && ~/.local/bin/uv pip install --system -q "cumulant_residual_cert@git+https://github.com/kootru-repo/cumulant-residual-cert.git" numpy


## Build a catalog and pick site assignments

Catalog and site assignments are independent of the data source.

In [ ]:
from cumulant_residual_cert import Catalog, certify, delta_ucb

cat = Catalog.chemistry_r4()
n_qubits = 4
sites_per_word = [
    (1, 2, 3),         # n_i n_j n_k
    (1, 2, 3),         # a_dag_i a_j n_k
    (1, 2, 3, 4),      # a_dag_i a_j n_k n_ell
    (1, 2, 3, 4),      # a_dag_i a_dag_j a_k a_ell
    (1, 2, 3, 4),      # n_i n_j n_k n_ell
]
print(f'n_qubits = {n_qubits}, |catalog| = {len(cat)}')

## The pluggable shadow fetcher

Each branch must return a list of `(basis, outcomes)` tuples in the contract format. The simulator branch is the only fully-implemented one; the others are skeletons that explain what you need to wire.

In [ ]:
import numpy as np

from cumulant_residual_cert.diagnostic import collect_shadows  # private test helper


def _two_particle_basis_state(n: int) -> np.ndarray:
    bits = [1, 1] + [0] * (n - 2)
    idx = 0
    for b in bits:
        idx = (idx << 1) | b
    psi = np.zeros(2 ** n, dtype=complex)
    psi[idx] = 1.0
    return np.outer(psi, psi.conj())


def fetch_shadow_data(provider: str, *, n_qubits: int, n_shots: int, **kwargs):
    """Return n_shots random-Pauli shadow records from the chosen provider.

    Each record is a (basis, outcomes) tuple where basis is a tuple of
    X/Y/Z labels of length n_qubits and outcomes is a tuple of +-1 of
    length n_qubits.
    """
    if provider == 'simulator':
        rho = kwargs.get('rho') if 'rho' in kwargs else _two_particle_basis_state(n_qubits)
        return collect_shadows(rho, n=n_qubits, M=n_shots, seed=kwargs.get('seed', 0))

    if provider == 'ibm':
        return _fetch_ibm(n_qubits=n_qubits, n_shots=n_shots, **kwargs)

    if provider == 'rigetti':
        return _fetch_rigetti(n_qubits=n_qubits, n_shots=n_shots, **kwargs)

    if provider == 'ionq':
        return _fetch_ionq(n_qubits=n_qubits, n_shots=n_shots, **kwargs)

    if provider == 'iqm':
        return _fetch_iqm(n_qubits=n_qubits, n_shots=n_shots, **kwargs)

    raise ValueError(
        f'unknown provider {provider!r}; available: simulator, ibm, rigetti, ionq, iqm'
    )

## Hardware-provider stubs

Each stub explains what you need to wire if you have the relevant account. None of them runs out of the box; they raise `NotImplementedError` until you fill them in.

**Range-factor note.** All four hardware-provider paths below produce random-Pauli shadow data, which carries the textbook $3^{|P|}$ Jordan-Wigner range factor. For chemistry-scale work at fermionic-word length $r \ge 3$, prefer **matchgate / fermionic-Gaussian shadows** via the OpenFermion adapter (in a later release of this library).

In [ ]:
def _fetch_ibm(*, n_qubits, n_shots, backend=None, token=None, **_):
    """IBM Quantum Runtime stub.

    Requires: uv add qiskit qiskit-ibm-runtime; an IBM Quantum account
    and API token.

    Outline:
      from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
      service = QiskitRuntimeService(channel='ibm_quantum', token=token)
      backend = backend or service.least_busy()
      sampler = SamplerV2(mode=backend)

    For each of n_shots, draw a random basis in {X, Y, Z}^n_qubits, build
    a Pauli-basis-rotation circuit, submit via sampler, and parse the bit
    string into a tuple of +-1 outcomes (b -> 1 - 2*b).

    Return: list of (basis, outcomes) tuples.
    """
    raise NotImplementedError('IBM path is a stub; wire qiskit-ibm-runtime here.')


def _fetch_rigetti(*, n_qubits, n_shots, qpu=None, **_):
    """Rigetti Forest stub. Requires pyquil + a QCS account."""
    raise NotImplementedError('Rigetti path is a stub; wire pyquil here.')


def _fetch_ionq(*, n_qubits, n_shots, api_key=None, **_):
    """IonQ stub. Requires the ionq-client package and an API key."""
    raise NotImplementedError('IonQ path is a stub; wire ionq-client here.')


def _fetch_iqm(*, n_qubits, n_shots, server_url=None, **_):
    """IQM Resonance / OpenQuantum stub.

    Requires the iqm-client (or openquantum-client) package and an IQM
    Resonance / OpenQuantum account.
    """
    raise NotImplementedError('IQM path is a stub; wire iqm-client here.')

## End-to-end on the simulator

This cell runs with no external dependencies.

In [ ]:
shots = fetch_shadow_data('simulator', n_qubits=n_qubits, n_shots=500, seed=42)
print(f'fetched {len(shots)} shadow shots from simulator')

ucb = delta_ucb(
    shadow_samples=shots,
    catalog=cat,
    sites_per_word=sites_per_word,
    n_qubits=n_qubits,
    confidence=0.95,
)
print(f'Delta UCB at 95% confidence: {ucb.delta_ucb:.4g}')
print(f'  n_paulis in Bonferroni union: {ucb.n_paulis}')
print()
for word, summary in ucb.per_word.items():
    print(f'  {word:32s}  ucb = {summary["ucb"]:.4g}  (kappa_hat = {summary["kappa_hat"]:.4g}, radius = {summary["radius"]:.4g})')

## Certify the truncation bias

Feed the diagnostic's $\Delta$ upper bound straight into `certify` to get the per-word bias bars.

In [ ]:
result = certify(cat, delta=ucb.delta_ucb)
for word, bar in result.bounds.items():
    print(f'  |tau({word})| <= {bar:.4g}')

## Switching providers

Once you have wired one of the hardware stubs, the rest of the notebook does not change:

```python
shots = fetch_shadow_data('ibm', n_qubits=n_qubits, n_shots=10_000, token='...')
ucb = delta_ucb(shadow_samples=shots, catalog=cat,
                sites_per_word=sites_per_word, n_qubits=n_qubits)
result = certify(cat, delta=ucb.delta_ucb)
```

The library is deliberately scoped to ingest shadow data and emit a certificate. Where the shots come from is your choice.